In [9]:
import pandas as pd
import torch
from pathlib import Path
from IPython.display import display

from baseline import (
    load_data,
    generate_labeled_subsets,
    load_labeled_subsets,
    create_loaders,
    train,
    LABEL_FRACTIONS,
    BATCH_SIZE,
    MAX_EPOCHS,
    PATIENCE,
    SEED,
    RESULTS_DIR,
    device,
)

In [7]:
train_df, val_df, test_df, image_paths, coco_images, annotations_by_image, = load_data()

print(f"Training images:   {len(train_df)}")
print(f"Validation images: {len(val_df)}")
print(f"Test images:       {len(test_df)}")

print("\nTraining class distribution:")
display(train_df["fractured"].value_counts().rename("count").to_frame())

print("\nValidation class distribution:")
display(val_df["fractured"].value_counts().rename("count").to_frame())

print("\nTest class distribution:")
display(test_df["fractured"].value_counts().rename("count").to_frame())

Training: 2856
Validation: 613
Test: 613
Indexed images: 4082
Training images:   2856
Validation images: 613
Test images:       613

Training class distribution:


,count
fractured,
0,2354
1,502



Validation class distribution:


,count
fractured,
0,505
1,108



Test class distribution:


,count
fractured,
0,505
1,108


In [4]:
generate_labeled_subsets(
    train_df=train_df,
    fractions=LABEL_FRACTIONS,
    seed=SEED,
)

Saved 10% subset: 285 images, 50 positive images
Saved 20% subset: 571 images, 100 positive images
Saved 30% subset: 857 images, 151 positive images
Saved 100% subset: 2856 images, 502 positive images


In [5]:
labeled_subsets = load_labeled_subsets(
    fractions=LABEL_FRACTIONS
)

for percentage, subset_df in labeled_subsets.items():
    print(
        f"{percentage}% labeled: "
        f"{len(subset_df)} images | "
        f"{int(subset_df['fractured'].sum())} fractured | "
        f"{int((subset_df['fractured'] == 0).sum())} non-fractured"
    )

100% labeled: 2856 images | 502 fractured | 2354 non-fractured
30% labeled: 857 images | 151 fractured | 706 non-fractured
20% labeled: 571 images | 100 fractured | 471 non-fractured
10% labeled: 285 images | 50 fractured | 235 non-fractured


In [4]:
distribution = []

for name, df in [
    ("Original Training", train_df),
    ("100%", labeled_subsets[100]),
    ("30%", labeled_subsets[30]),
    ("20%", labeled_subsets[20]),
    ("10%", labeled_subsets[10]),
]:
    distribution.append({
        "Dataset": name,
        "Total": len(df),
        "Fractured": (df["fractured"] == 1).sum(),
        "Non-fractured": (df["fractured"] == 0).sum(),
        "Fractured (%)": (df["fractured"] == 1).mean() * 100,
        "Non-fractured (%)": (df["fractured"] == 0).mean() * 100,
    })

distribution_df = pd.DataFrame(distribution)

display(distribution_df.round(2))

,Dataset,Total,Fractured,Non-fractured,Fractured (%),Non-fractured (%)
0,Original Training,2856,502,2354,17.58,82.42
1,100%,2856,502,2354,17.58,82.42
2,30%,857,151,706,17.62,82.38
3,20%,571,100,471,17.51,82.49
4,10%,285,50,235,17.54,82.46


In [7]:
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

experiment_results = []

for percentage in [100, 30, 20, 10]:

    print("\n" + "=" * 60)
    print(f"STARTING {percentage}% LABELED EXPERIMENT")
    print("=" * 60)

    # Get the corresponding labeled training subset
    subset_df = labeled_subsets[percentage]

    # Create data loaders
    train_loader, val_loader, test_loader = create_loaders(
        train_subset=subset_df,
        val_df=val_df,
        test_df=test_df,
        image_paths=image_paths,
        coco_images=coco_images,
        annotations_by_image=annotations_by_image,
        batch_size=BATCH_SIZE,
    )

    print(f"Training samples: {len(subset_df)}")
    print(f"Training batches: {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")

    # Train the model
    result = train(
        percentage=percentage,
        train_loader=train_loader,
        val_loader=val_loader,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
    )

    experiment_results.append(result)

    print(f"\nFinished {percentage}% experiment.")


STARTING 100% LABELED EXPERIMENT
Training samples: 2856
Training batches: 1428
Validation batches: 307


Training:   0%|          | 0/1428 [00:00<?, ?it/s]

Validation:   0%|          | 0/307 [00:00<?, ?it/s]

Epoch 01/20 | Train Loss: 1.1313 | Val Loss: 1.0224 | Positive Dice: 0.0001 | IoU: 0.8091 | Precision: 0.0167 | Recall: 0.0000 | 127.9s
Saved best checkpoint.


Training:   0%|          | 0/1428 [00:00<?, ?it/s]

Validation:   0%|          | 0/307 [00:00<?, ?it/s]

... (114 outputs left out)

Epoch 05/20 | Train Loss: 1.0813 | Val Loss: 1.0742 | Positive Dice: 0.0000 | IoU: 0.8238 | Precision: 1.0000 | Recall: 0.0000 | 22.3s


Training:   0%|          | 0/143 [00:00<?, ?it/s]

Validation:   0%|          | 0/307 [00:00<?, ?it/s]

Epoch 06/20 | Train Loss: 1.0623 | Val Loss: 1.0571 | Positive Dice: 0.0000 | IoU: 0.8059 | Precision: 0.0000 | Recall: 0.0000 | 21.8s
Early stopping after 5 epochs without improvement.

10% experiment complete. Best validation positive Dice: 0.0001

Finished 10% experiment.


In [8]:
results_df = pd.DataFrame(experiment_results)

display(results_df)

,percentage,best_val_positive_dice,checkpoint,history
0,100,0.389466,c:\Users\rylee\OneDrive\UCF\Independent Study\...,c:\Users\rylee\OneDrive\UCF\Independent Study\...
1,30,0.315606,c:\Users\rylee\OneDrive\UCF\Independent Study\...,c:\Users\rylee\OneDrive\UCF\Independent Study\...
2,20,0.000899,c:\Users\rylee\OneDrive\UCF\Independent Study\...,c:\Users\rylee\OneDrive\UCF\Independent Study\...
3,10,0.000064,c:\Users\rylee\OneDrive\UCF\Independent Study\...,c:\Users\rylee\OneDrive\UCF\Independent Study\...


In [9]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / "baseline_experiment_summary.csv"

results_df.to_csv(results_path, index=False)

print(f"Results saved to: {results_path}")

Results saved to: c:\Users\rylee\OneDrive\UCF\Independent Study\Bone_Fracture_Segmentation\processed\results\baseline_experiment_summary.csv


In [16]:
from pathlib import Path
import torch
import pandas as pd
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

from baseline import create_model, validate, create_loaders

checkpoint_dir = Path("processed/checkpoints")

test_results = []

for percentage in [100, 30, 20, 10]:

    checkpoint_path = (
        checkpoint_dir / f"unet_resnet18_{percentage}pct_best.pth"
    )

    print(f"\nTesting {percentage}% labeled model")

    # Create test loader
    _, _, test_loader = create_loaders(
        train_subset=labeled_subsets[percentage],
        val_df=val_df,
        test_df=test_df,
        image_paths=image_paths,
        coco_images=coco_images,
        annotations_by_image=annotations_by_image,
        batch_size=BATCH_SIZE,
    )

    # Load model weights
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model = create_model().to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Evaluate on test set
    metrics = validate(
        model=model,
        loader=test_loader
    )

    test_results.append({
        "fraction": percentage,
        **metrics,
    })

    print(metrics)

    del model, checkpoint, test_loader

    if device.type == "cuda":
        torch.cuda.empty_cache()

# Save results
test_results_df = pd.DataFrame(test_results)
display(test_results_df)



Testing 100% labeled model


Validation:   0%|          | 0/307 [00:00<?, ?it/s]

{'loss': 0.9484917018775255, 'dice': 0.28691923773596667, 'iou': 0.2699452663923112, 'positive_dice': 0.3507545585967786, 'positive_iou': 0.2544115545814009, 'precision': 0.15357949426065318, 'recall': 0.39968481397218164}

Testing 30% labeled model


Validation:   0%|          | 0/307 [00:00<?, ?it/s]

{'loss': 0.9592761304016798, 'dice': 0.5577412358469342, 'iou': 0.544684150462321, 'positive_dice': 0.2768090485119589, 'positive_iou': 0.20269799906040412, 'precision': 0.2641114929600194, 'recall': 0.26445934937742044}

Testing 20% labeled model


Validation:   0%|          | 0/307 [00:00<?, ?it/s]

{'loss': 1.1723982671345623, 'dice': 0.1975026927041484, 'iou': 0.19744772416358866, 'positive_dice': 0.0006401936123823448, 'positive_iou': 0.00032819624790885013, 'precision': 0.0030858118376791686, 'recall': 0.0003578198879335842}

Testing 10% labeled model


Validation:   0%|          | 0/307 [00:00<?, ?it/s]

{'loss': 1.3201593928780486, 'dice': 7.74054093857857e-07, 'iou': 3.8757673279079704e-07, 'positive_dice': 4.388370469447676e-06, 'positive_iou': 2.1947535959837157e-06, 'precision': 1.931005377849067e-06, 'recall': 7.613189850090511e-06}


,fraction,loss,dice,iou,positive_dice,positive_iou,precision,recall
0,100,0.948492,2.869192e-01,2.699453e-01,0.350755,0.254412,0.153579,0.399685
1,30,0.959276,5.577412e-01,5.446842e-01,0.276809,0.202698,0.264111,0.264459
2,20,1.172398,1.975027e-01,1.974477e-01,0.000640,0.000328,0.003086,0.000358
3,10,1.320159,7.740541e-07,3.875767e-07,0.000004,0.000002,0.000002,0.000008


In [15]:
test_results_df = pd.DataFrame(test_results)

display(test_results_df)

test_results_path = RESULTS_DIR / "baseline_test_results.csv"

test_results_df.to_csv(test_results_path, index=False)

print(f"Test results saved")

,fraction,loss,dice,iou,positive_dice,positive_iou,precision,recall
0,100,0.948492,2.869192e-01,2.699453e-01,0.350755,0.254412,0.153579,0.399685
1,30,0.959276,5.577412e-01,5.446842e-01,0.276809,0.202698,0.264111,0.264459
2,20,1.172398,1.975027e-01,1.974477e-01,0.000640,0.000328,0.003086,0.000358
3,10,1.320159,7.740541e-07,3.875767e-07,0.000004,0.000002,0.000002,0.000008


Test results saved
